In [1]:
import json
import math

with open("text_segmentation_dataset.json", "r") as f:
    data = json.load(f)

word_counts = data["word_counts"]
test_cases = data["test_cases"]

vocab = set(word_counts.keys())
total_words = data["metadata"]["total_corpus_words"]

### Greedy Approach

In [3]:
def greedy_segment(text, vocab):
    i = 0
    result = []

    while i < len(text):

        found = False
        for j in range(len(text), i, -1):

            word = text[i:j]

            if word in vocab:
                result.append(word)
                i = j
                found = True
                break

        # Unknown character
        if not found:
            result.append(text[i])
            i += 1

    return result

In [4]:
test_cases[0]['input']

'itthatthecitytakestepstothisproblem'

In [5]:
greedy_segment(test_cases[0]['input'], vocab)

['it',
 'that',
 'the',
 'city',
 'takes',
 't',
 'e',
 'p',
 's',
 'to',
 'this',
 'problem']

In [8]:
greedy_segment("likelywillthemillionissueearlierinthesessionashisfirst", vocab)

['likely',
 'will',
 'them',
 'i',
 'l',
 'l',
 'i',
 'on',
 'issue',
 'earlier',
 'in',
 'these',
 's',
 's',
 'i',
 'on',
 'as',
 'his',
 'first']

### Dynamic Programming Approach

In [22]:
## Log probabilities are used to avoid underflow issues when multiplying many small probabilities together. Instead of multiplying probabilities, we sum their logarithms.
log_probs = {}

for word, count in word_counts.items():
    log_probs[word] = math.log(count / total_words)

In [ ]:
import math


def dp_segment(text, word_counts):

    n = len(text)

    # dp[i] = best log probability to segment the words of text[0:i]
    dp = [-math.inf] * (n + 1)
    # parent[i] = previous position used to reach i
    parent = [-1] * (n + 1)
    dp[0] = 0

    for i in range(1, n + 1):
        for j in range(0, i):
            word = text[j:i]

            if word in word_counts:
                score = dp[j] + log_probs[word]
                if score > dp[i]:
                    dp[i] = score
                    parent[i] = j


    words = []
    i = n
    while i > 0:
        j = parent[i]

        if j == -1:
            words.append(text[i - 1])
            i -= 1
        else:
            word = text[j:i]
            words.append(word)
            i = j

    words.reverse()

    return words

In [27]:
dp_segment("likelywillthemillionissueearlierinthesessionashisfirst", word_counts)

['likely',
 'will',
 'the',
 'million',
 'issue',
 'earlier',
 'in',
 'the',
 'session',
 'as',
 'his',
 'first']

### Accuracy

In [9]:
def word_accuracy(predicted, actual):
  correct = 0

  for p, a in zip(predicted, actual):
    if p == a:
      correct += 1

  return correct / len(actual)

### Edit Distance

In [28]:
def edit_distance(predicted, actual):
  m = len(predicted)
  n = len(actual)

  dp = [[0] * (n + 1) for _ in range(m + 1)]  ## The minimum number of operations needed to convert the first i words of 'predicted' into the first j words of 'actual'

  for i in range(m + 1):
    dp[i][0] = i        ## We need i deletions to convert the first i words of 'predicted' into an empty string

  for j in range(n + 1):
    dp[0][j] = j        ## We need j insertions to convert an empty string into the first j words of 'actual'

  for i in range(1, m + 1):
    for j in range(1, n + 1):
      if predicted[i - 1] == actual[j - 1]:
        dp[i][j] = dp[i - 1][j - 1]

      else:
        dp[i][j] = 1 + min(  # Add 1 because we are performing an operation
          dp[i - 1][j],      # delete
          dp[i][j - 1],      # insert
          dp[i - 1][j - 1]   # substitute
        )

  return dp[m][n]

### Greedy Approach Evaluation

In [29]:
greedy_accuracy = 0
greedy_edit = 0

for sample in test_cases:
  inp = sample["input"]
  ground = sample["ground_truth"].split()
  prediction = greedy_segment(inp, vocab)
  greedy_accuracy += word_accuracy(prediction, ground)
  greedy_edit += edit_distance(prediction, ground)

greedy_accuracy /= len(test_cases)
greedy_edit /= len(test_cases)

print("Greedy Results")
print("----------------")
print("Accuracy :", greedy_accuracy)
print("Average Edit Distance :", greedy_edit)

Greedy Results
----------------
Accuracy : 0.8343584054834053
Average Edit Distance : 1.26


### Dynamic Programming Approach Evaluation

In [32]:
dp_accuracy = 0
dp_edit = 0

for sample in test_cases:
    inp = sample["input"]
    ground = sample["ground_truth"].split()
    prediction = dp_segment(inp, word_counts)
    dp_accuracy += word_accuracy(prediction, ground)
    dp_edit += edit_distance(prediction, ground)

dp_accuracy /= len(test_cases)
dp_edit /= len(test_cases)

print("\nDynamic Programming Results")
print("----------------------------")
print("Accuracy :", dp_accuracy)
print("Average Edit Distance :", dp_edit)


Dynamic Programming Results
----------------------------
Accuracy : 0.9903365079365078
Average Edit Distance : 0.037
